# Test a trained coastal segmentation model

Runs a trained YOLO26 semantic segmentation weight and saves a predicted class mask and a colored visualization for each one.

**Before running:** load this project onto Colab, and store your trained weight (e.g. `best.pt`) and test images in Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Load the project

Set `USE_GITHUB` below. The simplest method to load the project is cloning the GitHub repo.

In [ ]:
import os

USE_GITHUB = True

if USE_GITHUB:
    PROJECT_DIR = "/content/coastal-recession-ml"
    if not os.path.exists(PROJECT_DIR):
        !git clone https://github.com/lukasn9/coastal-recession-ml.git {PROJECT_DIR}
else:
    PROJECT_DIR = "/content/drive/MyDrive/coastal-recession-ml"

%cd {PROJECT_DIR}

In [ ]:
!pip install -q -r requirements.txt

## Load your trained model

Point `MODEL_PATH` at a trained weight, usually the `best.pt` saved during training. If you trained on Colab, this is the `weights/best.pt` file under the `project=` path you set in the training notebook.

`OUTPUT_DIR` is a project folder, not a single run's output. Each time you run inference below, results go into a new numbered subfolder inside it, `inference_1`, `inference_2`, and so on, so older results are never overwritten.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
from ultralytics import YOLO
from src.inference import next_run_dir, resolve_input, run_inference

MODEL_PATH = "/content/drive/MyDrive/coastal-recession-ml/runs/train/coastal-seg/weights/best.pt"
OUTPUT_DIR = "/content/drive/MyDrive/coastal-recession-ml/predictions"

model = YOLO(MODEL_PATH)

## Test a single image

Set `IMAGE_PATH` to one image and run this cell. It creates a new `inference_N` folder under `OUTPUT_DIR`, with a class mask in `masks/`, a colored visualization in `overlays/`, and a `results.csv` in `results/` with the class percentages for the image.

In [ ]:
IMAGE_PATH = "/content/drive/MyDrive/coastal-recession-ml/sample.png"

run_dir = next_run_dir(Path(OUTPUT_DIR))
summaries = run_inference(model, Path(IMAGE_PATH), run_dir)
for s in summaries:
    print(s)
print("Saved to", run_dir)

## Test a folder of images

Set `INPUT_DIR` to a folder, or a `.zip` file such as a Roboflow export, and run this cell. Zip files are unpacked automatically. If the resolved folder has a `test` subfolder inside it, images are read from there. Otherwise every image directly inside the folder is used. Results go into a new `inference_N` folder under `OUTPUT_DIR`, same as testing a single image, with one row per image in `results/results.csv`.

In [ ]:
INPUT_DIR = "/content/drive/MyDrive/coastal-recession-ml/test_images"

source = resolve_input(Path(INPUT_DIR))
run_dir = next_run_dir(Path(OUTPUT_DIR))
summaries = run_inference(model, source, run_dir)
for s in summaries:
    print(s)
print("Saved to", run_dir)

## Preview a result

Shows the last image from whichever cell you ran, colored by predicted class: blue for water, green for vegetation, brown for bare ground.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

first = summaries[0]
overlay_path = run_dir / "overlays" / f"{Path(first['file']).stem}.png"
overlay = Image.open(overlay_path)

plt.figure(figsize=(6, 6))
plt.imshow(overlay)
plt.axis("off")
plt.title(first["file"])
plt.show()

## Results table

Loads `results.csv` from the last run: one row per image, with the percentage of it taken up by each class.

In [ ]:
import pandas as pd

pd.read_csv(run_dir / "results" / "results.csv")